In [80]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()


True

In [81]:
client = OpenAI(
    api_key=os.environ['api_key'],
    base_url=os.environ['base_url'],
)


In [82]:
#定义调用函数的tool

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather", # 工具的名称： 小写、下划线， v开头
            "description": "Get weather of a location, the user should supply a location first.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city_name": {  # 入参属性名
                        "type": "string",  # 属性的类型： string、number、boolean
                        "description": "The city and state, e.g. San Francisco, CA",  #  参数的说明
                    }
                },
                "required": ["city_name"]  # 参数是否是必填的
            },
        }
    },
]

#定义函数
def get_weather(city_name):
    # 这里可以调用实际的天气API获取天气信息
    return f"{city_name}的天气是晴天，温度25摄氏度。"

In [83]:
#可用的函数列表
available_functions = {
    "get_weather": get_weather
}


In [84]:
message = [ {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "请告诉我北京的天气。"},]
response = client.chat.completions.create(
    model="deepseek-v4-pro",
    messages=message,
    tools=tools,
)


In [85]:
response

ChatCompletion(id='40c1f2e9-a426-4365-91b0-68a746cc1973', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_00_mfDZPJsA7q6UdzjtjXLi4369', function=Function(arguments='{"city_name": "北京"}', name='get_weather'), type='function', index=0)], reasoning_content='The user is asking for Beijing weather. I should call get_weather with city_name "北京" or "Beijing". The function requires city_name. I\'ll use "北京".'))], created=1790664804, model='deepseek-v4-pro', object='chat.completion', moderation=None, service_tier=None, system_fingerprint='a307abda487cd1b463329ccb945ce396', usage=CompletionUsage(completion_tokens=82, prompt_tokens=389, total_tokens=471, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=36, rejected_prediction_to

In [86]:
response.choices[0]. message.tool_calls[0].function

Function(arguments='{"city_name": "北京"}', name='get_weather')

In [87]:
response.choices[0]. message.tool_calls[0].function.arguments

'{"city_name": "北京"}'

In [88]:
message.append(response.choices[0].message.model_dump())
message
message.append({
    "role": "tool",   # 固定写法，表示这是工具调用的结果
    "tool_call_id": response.choices[0]. message.tool_calls[0].id,   # 关联到具体的工具调用id
    "content": str(get_weather("北京"))  # 工具执行的结果内容，必须是字符串格式
})

In [89]:
second_response = client.chat.completions.create(
    model="deepseek-v4-pro",
    messages=message
)

In [90]:
from IPython.display import Markdown
display(Markdown(second_response.choices[0].message.content))

根据最新数据，北京目前的天气是晴天，气温25摄氏度。天气不错，适合外出活动。